## 

# Load data

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer


# File name
RAW_DATA_FILE_NAME = "loan_data.csv"

PROJECT_ROOT = Path("..")
RAW_DATA_FILE_PATH = PROJECT_ROOT / "data" / "raw" / RAW_DATA_FILE_NAME

OUTPUT_DIR = PROJECT_ROOT / "data" / "pre_processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_FILE_PATH)
print("Output folder:", OUTPUT_DIR)

df = pd.read_csv(RAW_DATA_FILE_PATH)
print(df.shape)
df.head()

PROJECT_ROOT: ..
Raw data path: ../data/raw/loan_data.csv
Output folder: ../data/pre_processed
(45000, 14)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


# Data Preprocessing

In [ ]:
## Data col
target = "loan_status"

## Temp X_train, X_test
X = df.drop(columns=[target])
y = df[target].copy()
print("X shape:", X.shape)
print("y shape:", y.shape)

numeric_cols = [
    "person_age",
    "person_income",
    "person_emp_exp",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "credit_score"
]

onehot_cols = [
    "person_gender",
    "person_education",
    "person_home_ownership",
    "loan_intent"
]

binary_cols = ["previous_loan_defaults_on_file"]

## XGBoost pre processing pipeline
numeric_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_onehot = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Risk indicater for previous_loan_defaults_on_file: No -> 0, Yes -> 1
binary_encoder = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["No", "Yes"]]))
])

xgb_preprocessor = ColumnTransformer([
    ("num", numeric_xgb, numeric_cols),
    ("cat", categorical_onehot, onehot_cols),
    ("bin", binary_encoder, binary_cols)
])

## Neural Network pre processing pupeline
numeric_nn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) # tanh scaler
])

nn_preprocessor = ColumnTransformer([
    ("num", numeric_nn, numeric_cols),
    ("cat", categorical_onehot, onehot_cols),
    ("bin", binary_encoder, binary_cols)
])

X_xgb = xgb_preprocessor.fit_transform(X)
X_nn = nn_preprocessor.fit_transform(X)

xgb_feature_names = xgb_preprocessor.get_feature_names_out()
nn_feature_names = nn_preprocessor.get_feature_names_out()

xgb_df = pd.DataFrame(X_xgb, columns=xgb_feature_names, index=df.index)
nn_df = pd.DataFrame(X_nn, columns=nn_feature_names, index=df.index)

# Add target back so the output csv can be used directly for model training
xgb_df[target] = y.values
nn_df[target] = y.values

print("XGBoost preprocessed shape:", xgb_df.shape)
print("Neural Network preprocessed shape:", nn_df.shape)

xgb_df.head()

X shape: (45000, 13)
y shape: (45000,)
XGBoost preprocessed shape: (45000, 27)
Neural Network preprocessed shape: (45000, 27)


,num__person_age,num__person_income,num__person_emp_exp,num__loan_amnt,num__loan_int_rate,num__loan_percent_income,num__cb_person_cred_hist_length,num__credit_score,cat__person_gender_female,cat__person_gender_male,...,cat__person_home_ownership_OWN,cat__person_home_ownership_RENT,cat__loan_intent_DEBTCONSOLIDATION,cat__loan_intent_EDUCATION,cat__loan_intent_HOMEIMPROVEMENT,cat__loan_intent_MEDICAL,cat__loan_intent_PERSONAL,cat__loan_intent_VENTURE,bin__previous_loan_defaults_on_file,loan_status
0,22.0,71948.0,0.0,35000.0,16.02,0.49,3.0,561.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
1,21.0,12282.0,0.0,1000.0,11.14,0.08,2.0,504.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
2,25.0,12438.0,3.0,5500.0,12.87,0.44,3.0,635.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
3,23.0,79753.0,0.0,35000.0,15.23,0.44,2.0,675.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
4,24.0,66135.0,1.0,35000.0,14.27,0.53,4.0,586.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1


## Save preprocessed CSV files

In [7]:
xgb_output_path = OUTPUT_DIR / "loan_data_preprocessed_xgb.csv"
nn_output_path = OUTPUT_DIR / "loan_data_preprocessed_nn.csv"

xgb_df.to_csv(xgb_output_path, index=False)
nn_df.to_csv(nn_output_path, index=False)

print("Saved XGBoost data to:", xgb_output_path)
print("Saved Neural Network data to:", nn_output_path)

Saved XGBoost data to: ../data/pre_processed/loan_data_preprocessed_xgb.csv
Saved Neural Network data to: ../data/pre_processed/loan_data_preprocessed_nn.csv
